# Notebook 16 – Complete Preprocessing Workflow

## Question: How can we build a complete preprocessing workflow for a Machine Learning dataset?

This notebook demonstrates a complete preprocessing workflow using the Titanic dataset.

The workflow includes:

1. Load Dataset
2. Inspect Dataset
3. Identify Data Types
4. Identify Missing Values
5. Handle Missing Values
6. Detect Duplicates
7. Handle Duplicates
8. Validate Data
9. Detect Outliers
10. Treat Outliers
11. Encode Categorical Variables
12. Scale Numerical Variables
13. Handle Class Imbalance
14. Perform Feature Selection
15. Split Data
16. Build Preprocessing Pipeline
17. Validate Final Dataset

The final goal is to prepare a clean and ML-ready dataset while avoiding data leakage.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Question: What is the initial condition of the dataset?

The dataset is inspected to understand its structure, data types, missing values, and duplicate records before preprocessing.

In [2]:
print("Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (891, 12)

Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicate Rows: 0


## Question: What preprocessing problems were identified and how will they be handled?

### Problem
The Titanic dataset contains missing values, categorical variables, possible outliers, and columns that are not directly useful for Machine Learning.

### Analysis
These issues can affect model training, data quality, and model performance.

### Technique Selected
- Remove duplicate records
- Median imputation for numerical values
- Mode imputation for categorical values
- IQR-based outlier capping
- One-hot encoding for categorical variables
- Robust scaling for numerical variables
- Class weighting for imbalance
- Feature selection using SelectKBest
- Train-test split
- ColumnTransformer and Pipeline

### Reason
These techniques are suitable for the Titanic dataset and help preserve useful information while reducing the effect of data quality problems.

### Implementation
The preprocessing steps are implemented using Pandas and Scikit-learn.

### Result
The raw dataset is converted into a structured numerical dataset suitable for Machine Learning.

### Impact
A properly preprocessed dataset can improve model reliability, reduce unwanted bias from poor-quality data, and prevent data leakage.

## Question: How should duplicate records and invalid values be handled?

Duplicate records can cause the same observation to receive additional importance during training.

For the Titanic dataset, exact duplicate records are checked and removed.

Basic validation rules are also applied to important numerical and categorical columns.

In [3]:
df = df.drop_duplicates()

# Basic validation
print("Negative Age Values:", (df["Age"] < 0).sum())
print("Negative Fare Values:", (df["Fare"] < 0).sum())

print("\nSex Categories:")
print(df["Sex"].unique())

print("\nEmbarked Categories:")
print(df["Embarked"].dropna().unique())

print("\nShape after duplicate handling:", df.shape)

Negative Age Values: 0
Negative Fare Values: 0

Sex Categories:
<ArrowStringArray>
['male', 'female']
Length: 2, dtype: str

Embarked Categories:
<ArrowStringArray>
['S', 'C', 'Q']
Length: 3, dtype: str

Shape after duplicate handling: (891, 12)


## Question: How can missing values be handled?

`Age` is a numerical feature, so median imputation is selected because it is less affected by extreme values.

`Embarked` is categorical, so mode imputation is suitable.

`Cabin` contains a large amount of missing information and is not used in the final basic ML feature set.

The actual imputation will be performed inside the preprocessing pipeline using only the training data.

In [4]:
print("Missing values before preprocessing:")
print(df[["Age", "Fare", "Embarked"]].isnull().sum())

Missing values before preprocessing:
Age         177
Fare          0
Embarked      2
dtype: int64


## Question: How can outliers be detected before building the model?

The IQR method is used to identify potential outliers in numerical features such as `Age` and `Fare`.

An outlier is not automatically considered an error. Valid extreme observations should be retained where possible.

For the final workflow, outlier values will be capped using limits calculated from the training data.

In [5]:
for col in ["Age", "Fare"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    
    print(f"{col} Outliers:", outliers)

Age Outliers: 11
Fare Outliers: 116


## Question: How should the data be prepared before splitting?

The target variable is `Survived`.

Identification and text-heavy columns such as `PassengerId`, `Name`, `Ticket`, and `Cabin` are not used for the basic model.

The data is then divided into features (`X`) and target (`y`).

The split is performed before fitting preprocessing transformations so that information from the test data does not influence training.

In [6]:
from sklearn.model_selection import train_test_split

df_model = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

X = df_model.drop(columns=["Survived"])
y = df_model["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (712, 7)
Testing Shape: (179, 7)


## Question: How can missing values, outliers, encoding, and scaling be combined into a preprocessing pipeline?

A `ColumnTransformer` is used to apply different preprocessing steps to numerical and categorical features.

### Numerical Pipeline
- Median Imputation
- IQR-based outlier capping
- Robust Scaling

### Categorical Pipeline
- Most-frequent imputation
- One-Hot Encoding

The pipeline ensures that preprocessing is fitted only on the training data and then applied to the test data.

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

class IQRCapper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_ = X.quantile(0.25) - 1.5 * (X.quantile(0.75) - X.quantile(0.25))
        self.upper_ = X.quantile(0.75) + 1.5 * (X.quantile(0.75) - X.quantile(0.25))
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X = X.clip(self.lower_, self.upper_, axis=1)
        return X.values

numerical_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_capper", IQRCapper()),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (712, 10)
Processed Testing Shape: (179, 10)


## Question: How can class imbalance be handled?

The Titanic target contains two classes: `0` and `1`.

Instead of changing the original dataset through random over-sampling or under-sampling, class weights can be used during model training.

`class_weight="balanced"` gives more importance to the minority class and helps prevent the model from focusing mainly on the majority class.

In [9]:
print("Target Distribution:")
print(y.value_counts())

print("\nTarget Percentage:")
print(y.value_counts(normalize=True) * 100)

Target Distribution:
Survived
0    549
1    342
Name: count, dtype: int64

Target Percentage:
Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64


## Question: How can unnecessary features be reduced?

Feature selection removes less useful features and can reduce model complexity.

`SelectKBest` with mutual information is used to select the most informative features from the processed dataset.

In [10]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

k = min(8, X_train_processed.shape[1])

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=k
)

X_train_selected = selector.fit_transform(X_train_processed, y_train)
X_test_selected = selector.transform(X_test_processed)

print("Features Before Selection:", X_train_processed.shape[1])
print("Features After Selection:", X_train_selected.shape[1])

Features Before Selection: 10
Features After Selection: 8


## Question: How can the final dataset be validated?

The final dataset should contain:

- No missing values
- Numerical values only
- Consistent training and testing feature counts
- No duplicate rows introduced during preprocessing
- Features suitable for Machine Learning

The final validation confirms that the preprocessing workflow produced an ML-ready dataset.

In [11]:
print("Final Training Shape:", X_train_selected.shape)
print("Final Testing Shape:", X_test_selected.shape)

print("\nMissing Values in Training:", np.isnan(X_train_selected).sum())
print("Missing Values in Testing:", np.isnan(X_test_selected).sum())

print("\nFinal Data Type:", X_train_selected.dtype)

print("\nML-ready preprocessing completed successfully.")

Final Training Shape: (712, 8)
Final Testing Shape: (179, 8)

Missing Values in Training: 0
Missing Values in Testing: 0

Final Data Type: float64

ML-ready preprocessing completed successfully.


## Question: What is the final outcome of the complete preprocessing workflow?

The Titanic dataset was processed through a complete Machine Learning preprocessing workflow.

### Final Summary

- Dataset was loaded and inspected.
- Data types and missing values were identified.
- Duplicate records were checked and handled.
- Basic validation rules were applied.
- Outliers were detected using IQR.
- Outliers were capped inside the preprocessing pipeline.
- Missing values were imputed using appropriate strategies.
- Categorical variables were one-hot encoded.
- Numerical features were scaled using RobustScaler.
- Class imbalance was analyzed and can be handled using class weights.
- Feature selection was performed using mutual information.
- Data was split using stratified sampling.
- A preprocessing pipeline was created to avoid data leakage.
- The final data contains numerical features suitable for Machine Learning.

### Final Insight

A complete preprocessing workflow converts raw data into a consistent and ML-ready format while preserving useful information and reducing the risk of data leakage.